In [4]:
# 벡터 저장소가 이미 있는 상황
from dotenv import load_dotenv # api-key
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

In [5]:
# 1. 벡터스터오 가져오기

embedding = OpenAIEmbeddings(model="text-embedding-3-small")
persist_dir = "../7_vectorstore/samsung_2025_db"
collection_name = "samsung2025"

vectorstore = Chroma(
    persist_directory=persist_dir,
    collection_name=collection_name,
    embedding_function=embedding
)


In [6]:
# 2. retriever 만들기

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs = {
        "k":30
    }
)


In [7]:
# 3. reranker 만들기
from langchain_community.cross_encoders.huggingface import HuggingFaceCrossEncoder
from langchain.retrievers.document_compressors import CrossEncoderReranker

hf_cross_encoder = HuggingFaceCrossEncoder(
    model_name = "cross-encoder/ms-marco-MiniLM-L6-v2",
    model_kwargs={
        "device": "cuda",
        "max_length": 512
    }
)

reranker = CrossEncoderReranker(
    model = hf_cross_encoder,
    top_n = 10
)

In [8]:
# 4. retriever -> reranker 연결
from langchain.retrievers import ContextualCompressionRetriever
compress_retriever = ContextualCompressionRetriever(
    base_retriever=retriever,
    base_compressor=reranker
)

In [9]:
from langchain_community.document_transformers import LongContextReorder
# 5. reorder(순서 정리)
reorder = LongContextReorder()

In [10]:
from langchain_core.documents import Document
from typing import List
# 6. 검색 결과 문서 합치는 함수 생성
def format_docs(docs: List[Document]):
    return "\n\n".join([doc.page_content for doc in docs])

## 2. 기본 체인 만들기

In [19]:
# 1. 프롬프트 설정
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", """주어진 컨텍스트로만 근고로 간결하고 정확하게 답하도록 해라.
     [전문가]
     {pro}
     
     [컨텍스트]
     {context}
     """),
    ("human", "{question}"),
]
)

#2. 모델 설정
model = ChatOpenAI(
    model = "gpt-4.1-mini",
    temperature=0
)

# 3. outputparser
outputparser = StrOutputParser()

# 4. 체인 설정
chain = rag_prompt | model | outputparser

## 3. 통합 체인 만들기

In [12]:
rag_chain = (
    {
        "docs": RunnableLambda(lambda x: compress_retriever.invoke(x["question"])),
        "question": RunnablePassthrough()
    }
    | RunnableLambda(lambda x: {
        "context": format_docs(reorder.transform_documents(x["docs"])),
        "question": x["question"]
    })
    | chain
) 

rag_chain.invoke(
    {
        "question": "삼성의 미래 계획은 어떻게 되나요?"
    }
)

'삼성전자는 인재와 기술을 바탕으로 최고의 제품과 서비스를 창출하여 인류사회에 공헌한다는 경영철학 아래, 기술 리더십으로 재도약의 기반을 다지고 새로운 영역에서 미래 성장동력을 확보해 나갈 계획입니다. 또한, 2025년에는 청년 소프트웨어 인재 교육 기회를 마이스터고 졸업생까지 확대하고, 자립 준비 청년 지원 센터를 추가 설립하는 등 사회공헌 활동도 확대할 예정입니다.'

## 4. multi input chain

In [22]:
retriever_chain = RunnableLambda(lambda x: x['question']) | compress_retriever | format_docs

In [ ]:
rag_chain = (
    {
        "context": retriever_chain,
        "question": RunnablePassthrough()
    } | chain
)

rag_chain.invoke({"question": "삼성의 미래 계획은 어떻게 되나요?"})

'삼성전자는 인재와 기술을 바탕으로 최고의 제품과 서비스를 창출하여 인류사회에 공헌한다는 경영철학 아래, 기술 리더십으로 재도약의 기반을 다지고 새로운 영역에서 미래 성장동력을 확보해 나갈 계획입니다. 또한, 지속가능한 성장 기반 마련을 위해 이해관계자의 의견에 귀 기울이며 지속적으로 노력할 예정입니다.'

In [ ]:
rag_chain = (
    {
        "context": retriever_chain,
        "question": RunnablePassthrough(),
        "pro": RunnablePassthrough(lambda x: x['pro'])
    } | chain
)

rag_chain.invoke({"question": "삼성의 미래 계획은 어떻게 되나요?", "pro": "냥냥체"})

'삼성은 냥, 2050년까지 탄소중립을 목표로 환경경영을 쭉 이어갈 냥! 청년 교육도 팡팡 지원해서 SW·AI 인재도 키우고, 중소기업과 스타트업도 활짝 도울 계획이래냥~ 미래 기술과 사람 모두 챙기는 멋진 계획이라냥!'

: 